In [1]:
# ============================================================
# CELL 10 — MODEL 2 INFERENCE TEST
# ============================================================

import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# ------------------------------------------------------------
# 1. Define the saved model path
# ------------------------------------------------------------

BASE_DIR = os.getcwd()

FINAL_MODEL_DIR = os.path.join(
    BASE_DIR,
    "outputs",
    "model2_clinicalbert",
    "final"
)

# ------------------------------------------------------------
# 2. Load saved tokenizer and model
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

model = AutoModelForSequenceClassification.from_pretrained(
    FINAL_MODEL_DIR
)

# ------------------------------------------------------------
# 3. Use GPU if available
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

# ------------------------------------------------------------
# 4. Check that everything loaded correctly
# ------------------------------------------------------------

print("Model loaded successfully!")
print("Device:", device)
print("Number of labels:", model.config.num_labels)
print("Model path:", FINAL_MODEL_DIR)

c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15579.53it/s]


Model loaded successfully!
Device: cuda
Number of labels: 36
Model path: c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\outputs\model2_clinicalbert\final


In [2]:
# ============================================================
# CELL 11 — SINGLE CONVERSATION PREDICTION
# ============================================================

conversation = """
User: I've had stomach pain since yesterday.
Assistant: Where exactly does it hurt?
User: Mostly on the right side.
"""

# Tokenize conversation
inputs = tokenizer(
    conversation,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Move inputs to GPU/CPU
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# Model prediction
with torch.no_grad():
    outputs = model(**inputs)

# Get predicted class
predicted_id = torch.argmax(
    outputs.logits,
    dim=-1
).item()

predicted_label = model.config.id2label[predicted_id]

# Display result
print("=" * 60)
print("MODEL 2 INFERENCE RESULT")
print("=" * 60)

print("\nConversation:")
print(conversation)

print("Predicted label ID:", predicted_id)
print("Predicted label:", predicted_label)

MODEL 2 INFERENCE RESULT

Conversation:

User: I've had stomach pain since yesterday.
Assistant: Where exactly does it hurt?
User: Mostly on the right side.

Predicted label ID: 1
Predicted label: associated_symptoms


In [3]:
# ============================================================
# CELL 12 — UNSEEN CONVERSATION TEST
# ============================================================

conversation = """
User: My stomach has been bothering me for about three days.
Assistant: Can you tell me where in your abdomen you feel it?
User: It's mostly around the upper middle area.
Assistant: What does the discomfort feel like?
User: It's more of a burning sensation, especially after I eat.
"""

inputs = tokenizer(
    conversation,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = model(**inputs)

# Convert logits to probabilities
probabilities = torch.softmax(outputs.logits, dim=-1)

predicted_id = torch.argmax(
    probabilities,
    dim=-1
).item()

predicted_label = model.config.id2label[predicted_id]
confidence = probabilities[0, predicted_id].item()

print("=" * 60)
print("UNSEEN CONVERSATION TEST")
print("=" * 60)

print("\nConversation:")
print(conversation)

print("Predicted label:", predicted_label)
print("Confidence:", round(confidence * 100, 2), "%")

UNSEEN CONVERSATION TEST

Conversation:

User: My stomach has been bothering me for about three days.
Assistant: Can you tell me where in your abdomen you feel it?
User: It's mostly around the upper middle area.
Assistant: What does the discomfort feel like?
User: It's more of a burning sensation, especially after I eat.

Predicted label: associated_symptoms
Confidence: 23.88 %


In [5]:
# ============================================================
# CELL 13 — MULTIPLE UNSEEN CONVERSATION TESTS
# ============================================================

test_cases = [
    {
        "name": "Test 1 - Abdominal pain quality",
        "conversation": """
User: I've been having an ache in my abdomen since this morning.
Assistant: Where exactly is the pain?
User: Around the lower right side.
""",
        "expected": "pain_quality"
    },

    {
        "name": "Test 2 - Abdominal pain severity",
        "conversation": """
User: My abdomen has been hurting since last night.
Assistant: Where do you feel the pain?
User: Mostly in the center.
Assistant: What does the pain feel like?
User: It's a sharp pain.
""",
        "expected": "pain_severity"
    },

    {
        "name": "Test 3 - Cough character",
        "conversation": """
User: I've been coughing for two days.
Assistant: How often are you coughing?
User: Several times throughout the day.
""",
        "expected": "cough_character"
    },

    {
        "name": "Test 4 - Dizziness onset",
        "conversation": """
User: I've been feeling dizzy.
Assistant: How frequently does the dizziness happen?
User: It comes and goes a few times each day.
""",
        "expected": "dizziness_onset"
    },

    {
        "name": "Test 5 - Fever temperature",
        "conversation": """
User: I've had a fever since yesterday.
Assistant: How has the fever changed?
User: It seems to be getting worse.
""",
        "expected": "fever_temperature"
    },

    {
        "name": "Test 6 - Vomiting frequency",
        "conversation": """
User: I've been throwing up since this morning.
Assistant: When did the vomiting start?
User: About six hours ago.
""",
        "expected": "vomiting_frequency"
    },

    {
        "name": "Test 7 - Rash character",
        "conversation": """
User: I noticed a rash on my arms.
Assistant: Has the rash changed over time?
User: It has been spreading slowly.
""",
        "expected": "rash_character"
    },

    {
        "name": "Test 8 - Breathing severity",
        "conversation": """
User: I've been having trouble breathing since yesterday.
Assistant: When did the breathing problem begin?
User: Yesterday afternoon.
Assistant: Has it changed since then?
User: It's getting worse.
""",
        "expected": "breath_severity"
    },

    {
        "name": "Test 9 - Urinary onset",
        "conversation": """
User: I've been having some urinary problems.
Assistant: How frequently are you urinating?
User: Much more often than usual.
""",
        "expected": "urinary_onset"
    },

    {
        "name": "Test 10 - Associated symptoms",
        "conversation": """
User: My stomach has been hurting for two days.
Assistant: Where does it hurt?
User: Around the middle of my abdomen.
Assistant: What does the pain feel like?
User: It's a dull ache.
Assistant: How severe is it?
User: About a five out of ten.
""",
        "expected": "associated_symptoms"
    }
]


# ------------------------------------------------------------
# Run all tests
# ------------------------------------------------------------

results = []

for test in test_cases:

    inputs = tokenizer(
        test["conversation"],
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # Top 3 predictions
    top_probs, top_ids = torch.topk(
        probabilities,
        k=3
    )

    top_predictions = []

    for prob, idx in zip(top_probs, top_ids):
        label = model.config.id2label[idx.item()]
        top_predictions.append(
            f"{label} ({prob.item() * 100:.2f}%)"
        )

    predicted_label = model.config.id2label[
        top_ids[0].item()
    ]

    correct = predicted_label == test["expected"]

    results.append({
        "name": test["name"],
        "expected": test["expected"],
        "predicted": predicted_label,
        "correct": correct,
        "top_3": top_predictions
    })


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 80)
print("MODEL 2 — UNSEEN TEST RESULTS")
print("=" * 80)

for result in results:

    print("\n" + result["name"])
    print("-" * 80)

    print("Expected :", result["expected"])
    print("Predicted:", result["predicted"])
    print("Correct  :", "YES" if result["correct"] else "NO")

    print("\nTop 3 predictions:")

    for prediction in result["top_3"]:
        print("  ", prediction)

print("\n" + "=" * 80)

correct_count = sum(
    result["correct"]
    for result in results
)

print(
    f"Overall: {correct_count}/{len(results)} "
    f"correct"
)

print("=" * 80)

MODEL 2 — UNSEEN TEST RESULTS

Test 1 - Abdominal pain quality
--------------------------------------------------------------------------------
Expected : pain_quality
Predicted: associated_symptoms
Correct  : NO

Top 3 predictions:
   associated_symptoms (24.80%)
   pain_quality (7.14%)
   functional_impact (5.00%)

Test 2 - Abdominal pain severity
--------------------------------------------------------------------------------
Expected : pain_severity
Predicted: associated_symptoms
Correct  : NO

Top 3 predictions:
   associated_symptoms (25.27%)
   pain_quality (6.71%)
   functional_impact (5.14%)

Test 3 - Cough character
--------------------------------------------------------------------------------
Expected : cough_character
Predicted: associated_symptoms
Correct  : NO

Top 3 predictions:
   associated_symptoms (27.13%)
   pain_quality (6.10%)
   functional_impact (4.91%)

Test 4 - Dizziness onset
--------------------------------------------------------------------------------
E

In [6]:
# ============================================================
# CELL 14 — CHECK MODEL 2 LABEL DISTRIBUTION
# ============================================================

from datasets import load_from_disk
import os
from collections import Counter

PREPROCESSED_DIR = os.path.join(
    BASE_DIR,
    "model2_preprocessed"
)

DATASET_PATH = os.path.join(
    PREPROCESSED_DIR,
    "dataset"
)

dataset = load_from_disk(DATASET_PATH)

# Count how many examples belong to each label
label_counts = Counter(dataset["label_name"])

print("=" * 60)
print("MODEL 2 LABEL DISTRIBUTION")
print("=" * 60)

for label, count in sorted(
    label_counts.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{label:30s} : {count}")

print("=" * 60)

print("Total examples:", len(dataset))
print("Total labels:", len(label_counts))

print("\nMost frequent label:")
print(
    label_counts.most_common(1)[0]
)

print("\nLeast frequent labels:")

minimum_count = min(label_counts.values())

for label, count in sorted(label_counts.items()):
    if count == minimum_count:
        print(f"{label:30s} : {count}")

MODEL 2 LABEL DISTRIBUTION
associated_symptoms            : 25
RED_FLAG                       : 15
functional_impact              : 8
pain_location                  : 7
pain_severity                  : 6
pain_quality                   : 5
injury_or_trigger              : 3
pain_progression               : 2
cough_character                : 2
pain_triggers                  : 1
pain_relieving_factors         : 1
cough_frequency                : 1
cough_progression              : 1
dizziness_onset                : 1
dizziness_frequency            : 1
dizziness_progression          : 1
dizziness_triggers             : 1
dizziness_type                 : 1
throat_severity                : 1
fever_temperature              : 1
fever_progression              : 1
nausea_onset                   : 1
vomiting                       : 1
vomiting_frequency             : 1
diarrhea_frequency             : 1
severity                       : 1
duration                       : 1
breath_onset              